In [3]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer


class EEGTextMetaDataset(Dataset):
    def __init__(self, eeg_dir, metadata_dir, tokenizer, max_length=64, use_emotional_tone=True):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_emotional_tone = use_emotional_tone

        # -------------------------
        # 1. Load EEG files (all subjects)
        # -------------------------
        eeg_files = []
        for root, dirs, files in os.walk(eeg_dir):
            for f in files:
                if f.endswith(".npy") and "_preprocessed" in f:
                    eeg_files.append(os.path.join(root, f))
        eeg_files = sorted(eeg_files)

        if not eeg_files:
            raise FileNotFoundError(f"No EEG .npy files found in {eeg_dir}")

        self.eeg_file_paths = eeg_files
        self.eeg_data_list = []
        self.index_map = []

        for subj_idx, path in enumerate(self.eeg_file_paths):
            eeg = np.load(path, mmap_mode='r')
            assert eeg.ndim == 3 and eeg.shape[1:] == (62, 400), \
                f"EEG file {path} has shape {eeg.shape}, expected (*, 62, 400)"
            self.eeg_data_list.append(eeg)
            n_samples = eeg.shape[0]
            self.index_map.extend([(subj_idx, i) for i in range(n_samples)])

        total_samples = len(self.index_map)
        print(f"Found {len(self.eeg_file_paths)} EEG files → Total samples: {total_samples}")

        # -------------------------
        # 2. Load Metadata JSONs
        # -------------------------
        metadata_files = sorted(
            [os.path.join(dp, f)
             for dp, dn, filenames in os.walk(metadata_dir)
             for f in filenames if f.endswith(".json")]
        )
        if not metadata_files:
            raise FileNotFoundError(f"No metadata JSON files found in {metadata_dir}")

        self.metadata_list = []
        for fpath in metadata_files:
            with open(fpath, 'r', encoding='utf-8') as f:
                meta = json.load(f)

                # --- Assertion for essential content ---
                assert "semantic_features" in meta and "scene_category" in meta["semantic_features"], \
                    f"Missing scene_category in {fpath}"
                assert "visual_attributes" in meta and "major_colors" in meta["visual_attributes"], \
                    f"Missing major_colors in {fpath}"
                assert "optical_flow_score" in meta["visual_attributes"], \
                    f"Missing optical_flow_score in {fpath}"
                
                # Robustly handle missing 'objects' key
                if "objects" not in meta.get("semantic_features", {}):
                    meta["semantic_features"]["objects"] = []

                self.metadata_list.append(meta)

        base_count = len(self.metadata_list)
        print(f"Loaded {base_count} metadata JSON files")

        # -------------------------
        # 3. Build base captions
        # -------------------------
        base_captions = []
        for meta in self.metadata_list:
            caption_text = meta["caption"]["text"]
            if self.use_emotional_tone and "emotional_tone" in meta["caption"]:
                caption_text += f". Tone: {meta['caption']['emotional_tone']}"
            base_captions.append(caption_text)

        # Repeat for each subject
        num_subjects = len(self.eeg_file_paths)
        self.captions = base_captions * num_subjects
        self.metadata_repeated = self.metadata_list * num_subjects

        assert len(self.captions) == len(self.metadata_repeated) == len(self.index_map), \
            "Mismatch after repeating captions and metadata for subjects"

        # -------------------------
        # 4. Encode metadata categories (Scene, Color, and Objects)
        # -------------------------
        scene_categories = sorted(list({m["semantic_features"]["scene_category"] for m in self.metadata_list}))
        colors = sorted(list({m["visual_attributes"]["major_colors"][0]["color"].split()[0]
                               for m in self.metadata_list}))
        all_objects = set()
        for m in self.metadata_list:
            all_objects.update(m["semantic_features"]["objects"])
        objects_vocab = sorted(list(all_objects))

        self.scene_to_id = {scene: i for i, scene in enumerate(scene_categories)}
        self.color_to_id = {c: i for i, c in enumerate(colors)}
        self.object_to_id = {obj: i for i, obj in enumerate(objects_vocab)}
        
        # Create inverse mapping for easier lookup
        self.id_to_scene = {i: scene for scene, i in self.scene_to_id.items()}
        self.id_to_color = {i: c for c, i in self.color_to_id.items()}
        self.id_to_object = {i: obj for obj, i in self.object_to_id.items()}

        print(f"Scene categories: {len(self.scene_to_id)} | Colors: {len(self.color_to_id)} | Objects: {len(self.object_to_id)}")

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        subj_idx, local_idx = self.index_map[idx]
        eeg_tensor = torch.tensor(self.eeg_data_list[subj_idx][local_idx], dtype=torch.float32)
        caption = self.captions[idx]
        tokenized = self.tokenizer(
            caption, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        tokenized = {k: v.squeeze(0) for k, v in tokenized.items()}
        meta = self.metadata_repeated[idx]
        
        scene_id = self.scene_to_id[meta["semantic_features"]["scene_category"]]
        color_id = self.color_to_id[meta["visual_attributes"]["major_colors"][0]["color"].split()[0]]
        motion_score = float(meta["visual_attributes"]["optical_flow_score"]["value"])
        scalar_meta_tensor = torch.tensor([scene_id, color_id, motion_score], dtype=torch.float32)

        objects_present = meta["semantic_features"]["objects"]
        object_multi_hot = torch.zeros(len(self.object_to_id), dtype=torch.float32)
        for obj in objects_present:
            if obj in self.object_to_id:
                obj_id = self.object_to_id[obj]
                object_multi_hot[obj_id] = 1.0
        
        metadata_tensor = torch.cat((scalar_meta_tensor, object_multi_hot))
        return eeg_tensor, tokenized, metadata_tensor

# -------------------------------
# Test the dataset
# -------------------------------
if __name__ == "__main__":
    tokenizer = BertTokenizer.from_pretrained("/home/poorna/models/bert-base-uncased")

    EEG_DIR = "/home/poorna/data/preprocessed_eeg"
    METADATA_DIR = "/home/poorna/data/metadata_dir"

    dataset = EEGTextMetaDataset(
        eeg_dir=EEG_DIR,
        metadata_dir=METADATA_DIR,
        tokenizer=tokenizer,
        max_length=64
    )

    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    for eeg_batch, tokenized_batch, meta_batch in loader:
        print("\nEEG batch:", eeg_batch.shape)
        print("Input IDs:", tokenized_batch["input_ids"].shape)
        print("Metadata batch:", meta_batch.shape)

        # Sample inspection
        sample_idx = 0
        print("\nSample Inspection:")
        print("EEG shape:", eeg_batch[sample_idx].shape)
        print("Token IDs:", tokenized_batch["input_ids"][sample_idx][:20])
        print("Decoded caption:", tokenizer.decode(
            tokenized_batch["input_ids"][sample_idx], skip_special_tokens=True
        ))
        print("Metadata tensor:", meta_batch[sample_idx])

        # --- Decode all parts of the metadata tensor ---
        sample_meta_tensor = meta_batch[sample_idx]
        
        scene_id = int(sample_meta_tensor[0].item())
        color_id = int(sample_meta_tensor[1].item())
        motion_score = float(sample_meta_tensor[2].item())
        
        # Use efficient pre-computed maps for lookup
        scene_name = dataset.id_to_scene.get(scene_id, "Unknown")
        color_name = dataset.id_to_color.get(color_id, "Unknown")
        
        # Decode the object vector part of the tensor
        object_vector = sample_meta_tensor[3:]
        present_object_ids = torch.where(object_vector == 1.0)[0]
        present_objects = [dataset.id_to_object.get(i.item(), "Unknown") for i in present_object_ids]

        print(f"Scene: {scene_name}, Color: {color_name}, Motion: {motion_score:.3f}")
        print(f"Objects: {present_objects}") # Added to show decoded objects

        break

    print(f"\nTotal samples loaded: {len(dataset)}")

Found 20 EEG files → Total samples: 28000
Loaded 1400 metadata JSON files
Scene categories: 77 | Colors: 53 | Objects: 1013

EEG batch: torch.Size([32, 62, 400])
Input IDs: torch.Size([32, 64])
Metadata batch: torch.Size([32, 1016])

Sample Inspection:
EEG shape: torch.Size([62, 400])
Token IDs: tensor([  101,  2402,  2450,  3248,  6490,  2858,  2006,  1037, 23308, 12549,
         1037,  2103,  1012,  1012,  4309,  1024,  5475,   102,     0,     0])
Decoded caption: young woman plays acoustic guitar on a rooftop overlooking a city.. tone : calm
Metadata tensor: tensor([68.0000, 51.0000,  0.1000,  ...,  0.0000,  0.0000,  0.0000])
Scene: urban, Color: yellow, Motion: 0.100
Objects: ['acoustic guitar', 'buildings', 'city', 'rooftop', 'woman']

Total samples loaded: 28000


In [6]:
import h5py
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer

# This assumes the EEGTextMetaDataset class definition is in the same file or imported
# from the previous step.

# --- Paths ---
HDF5_FILE = "/home/poorna/data/eeg_dataset_with_objects.h5"
EEG_DIR = "/home/poorna/data/preprocessed_eeg"
METADATA_DIR = "/home/poorna/data/metadata_dir"

# --- Dataset ---
print("Initializing dataset...")
tokenizer = BertTokenizer.from_pretrained("/home/poorna/models/bert-base-uncased")
dataset = EEGTextMetaDataset(EEG_DIR, METADATA_DIR, tokenizer)
loader = DataLoader(dataset, batch_size=1, shuffle=False)  # batch=1 for sequential save

# --- Get dynamic shapes from a sample ---
# This makes the code robust to changes in the dataset's output shapes.
n_samples = len(dataset)
sample_eeg, sample_tokenized, sample_meta = dataset[0]

# --- Create HDF5 file ---
print(f"Creating HDF5 file at {HDF5_FILE}...")
with h5py.File(HDF5_FILE, "w") as f:
    # <<< MODIFIED: Shapes are now determined dynamically from the dataset >>>
    eeg_shape = (n_samples, *sample_eeg.shape)
    token_shape = (n_samples, *sample_tokenized["input_ids"].shape)
    # This correctly gets the full metadata shape (e.g., 3 + 1013 = 1016)
    meta_shape = (n_samples, *sample_meta.shape)

    print(f"Allocating space for {n_samples} samples...")
    print(f"  - EEG shape: {eeg_shape}")
    print(f"  - Token shape: {token_shape}")
    print(f"  - Metadata shape: {meta_shape}")

    eeg_ds = f.create_dataset("eeg", shape=eeg_shape, dtype="float32")
    tokens_ds = f.create_dataset("input_ids", shape=token_shape, dtype="int64")
    meta_ds = f.create_dataset("metadata", shape=meta_shape, dtype="float32")

    # Iterate and save directly to file (low memory usage)
    print("Writing data to HDF5 file...")
    for idx, (eeg_tensor, tokenized, metadata_tensor) in enumerate(loader):
        if (idx + 1) % 1000 == 0:
            print(f"  ... processed {idx + 1}/{n_samples} samples")
        
        eeg_ds[idx] = eeg_tensor.squeeze(0).numpy()
        tokens_ds[idx] = tokenized["input_ids"].squeeze(0).numpy()
        meta_ds[idx] = metadata_tensor.squeeze(0).numpy() # This will now work correctly

print(f"\nSuccessfully saved dataset to {HDF5_FILE}")

Initializing dataset...
Found 20 EEG files → Total samples: 28000
Loaded 1400 metadata JSON files
Scene categories: 77 | Colors: 53 | Objects: 1013
Creating HDF5 file at /home/poorna/data/eeg_dataset_with_objects.h5...
Allocating space for 28000 samples...
  - EEG shape: (28000, 62, 400)
  - Token shape: (28000, 64)
  - Metadata shape: (28000, 1016)
Writing data to HDF5 file...
  ... processed 1000/28000 samples
  ... processed 2000/28000 samples
  ... processed 3000/28000 samples
  ... processed 4000/28000 samples
  ... processed 5000/28000 samples
  ... processed 6000/28000 samples
  ... processed 7000/28000 samples
  ... processed 8000/28000 samples
  ... processed 9000/28000 samples
  ... processed 10000/28000 samples
  ... processed 11000/28000 samples
  ... processed 12000/28000 samples
  ... processed 13000/28000 samples
  ... processed 14000/28000 samples
  ... processed 15000/28000 samples
  ... processed 16000/28000 samples
  ... processed 17000/28000 samples
  ... processed 

In [7]:
import h5py
import torch

with h5py.File("/home/poorna/data/eeg_dataset_with_objects.h5", "r") as f:
    print(list(f.keys()))  # ['eeg', 'input_ids', 'metadata']
    eeg_sample = torch.tensor(f["eeg"][0])         # first sample EEG tensor
    tokens_sample = torch.tensor(f["input_ids"][0])
    meta_sample = torch.tensor(f["metadata"][0])

print(eeg_sample.shape, tokens_sample.shape, meta_sample.shape)

['eeg', 'input_ids', 'metadata']
torch.Size([62, 400]) torch.Size([64]) torch.Size([1016])


In [11]:
import h5py

HDF5_FILE = "/home/poorna/data/eeg_dataset_with_objects.h5"

with h5py.File(HDF5_FILE, "r") as f:
    print("\nDatasets in file:", list(f.keys()))

    for name in f.keys():
        dset = f[name]
        print(f"{name}: shape={dset.shape}, dtype={dset.dtype}")

    # Optional: verify a few entries
    sample_idx = 0
    print("\nSample check:")
    print("EEG sample shape:", f["eeg"][sample_idx].shape)
    print("Tokens sample shape:", f["input_ids"][sample_idx].shape)
    print("Metadata sample shape:", f["metadata"][sample_idx].shape)


Datasets in file: ['eeg', 'input_ids', 'metadata']
eeg: shape=(28000, 62, 400), dtype=float32
input_ids: shape=(28000, 64), dtype=int64
metadata: shape=(28000, 1016), dtype=float32

Sample check:
EEG sample shape: (62, 400)
Tokens sample shape: (64,)
Metadata sample shape: (1016,)
